In [2]:
from astropy.io import fits
from astropy.table import Table
import matplotlib.pyplot as plt
import numpy as np
from pykoa.koa import Koa 
import os
import time
import pandas as pd

!pip install pandas




In [20]:
# Used to delete directories.

# import shutil
# shutil.rmtree('dnload_dir_hires_calib1/WASP-96')

In [2]:
# Import the Target List file and extract the hostnames for each star, which will be used to get the data for each star using Koa

filename = 'C:/Users/Aidan Moran-Bates/Downloads/ASTR502_Master_Target_List.csv'


#### Checks to see if the Target List was imported successfully
#df = pd.read_csv(filename)
#print(df)

# Extracts the 2nd column of the Target List which has the host names for each star
df_column_2 = pd.read_csv(filename, usecols=[1])

# Checks to see if the host names were extracted successfully.
print(df_column_2)



      hostname
0     HIP 65 A
1     WASP-136
2       KELT-1
3      HATS-34
4      WASP-96
...        ...
4511   Qatar-3
4512  WASP-147
4513    WASP-5
4514  TOI-3629
4515    WASP-8

[4516 rows x 1 columns]


In [3]:
# Turns the column of star host names into an array which
# will then be turned into a list that can be easily used in an iterable function

host_name_array = np.array(df_column_2)
host_name_list = [item[0] for item in host_name_array]

### Testing that an iterable function can access each part of the list.
# for i in range(len(host_name_list)):
#     dd = (host_name_list[i])
#     print((dd))


##  Iterable Function Works!!

In [4]:
#help(Koa) # Produces information on how Koa works.

### Below is information on how to acquire and download fits files of stars.

# instrument = 'hires'
# date_range = '2018-03-16 00:00:00/2018-03-18 00:00:00'
# output_meta_path = './metadata.tbl'
# output_format = 'ipac'
# download_dir = './fits_downloads'

# # Create download directory if it doesn't exist
# os.makedirs(download_dir, exist_ok=True)

# # --- 1. Query KOA and save metadata ---
# print(f"Querying KOA for {instrument} data within {date_range}...")
# Koa.query_datetime(instrument, date_range, outpath=output_meta_path, format=output_format)
# print(f"Metadata saved to {output_meta_path}")

# # --- 2. Download FITS files ---
# print(f"Starting download to {download_dir}...")
# Koa.download(metapath=output_meta_path, format=output_format, outdir=download_dir)

In [ ]:
# Creates directory 'output',   once directory is created code will say directory already exists.
# This directory will be where the .tbl files for each star will be downloaded (these files hold the fits files)
try:
    os.mkdir('./output')
except:
    print(" Directory exists already", flush=True)

In [29]:

# Gets fits for a specific star from the "hires" instrument and downloads it into a .tbl
# Is tested using one of the first stars in the target list "HIP 65 A"
Koa.query_object ('hires', \
                  'HIP 65 A, \
                  './output/HIP_65_A.tbl', overwrite=True,)

rec = Table.read ('./output/HIP_65_A.tbl', format='ascii.ipac')
print (rec)

object name resolved: ra= 0.1856063, dec=-54.8308228
submitting request...
Result downloaded to file [./output/HIP_65_A.tbl]
koaid ofname instrume targname object ... equinox datlevel outfile filehand
----- ------ -------- -------- ------ ... ------- -------- ------- --------


In [5]:
# Takes fits files downloaded from star search .tbl files and gets the lvl 1 data which it puts into the "dnload_dir_hires_calib1" folder
# Was tested using star HAT-P-16 which is one of the first stars from the target list.
Koa.download ('./output/HAT-P-16.tbl', \
    'ipac', \
    'dnload_dir_hires_calib1/HAT-P-16', \
    start_row=53, \
    end_row=55, \
    lev1file=1 )

Start downloading -18 koaid data you requested;
please check your outdir: dnload_dir_hires_calib1/HAT-P-16 for  progress ....

A total of 0 new lev0 FITS files downloaded.


In [22]:
# Full iterable function combining the code of the two cells above with the iterable function of "host_name_list".
# Code is deigned to get the .tbl file for each star and then extract the level 1 data from each .tbl file

# Variable that I will manually change whenever I take breaks from downloading star data so code doesn't start back from beginning
# Just completed star i = 34
progress_jump = 0  # If running code for first time this should be zero

for i in range(len(host_name_list)):
    j = i + progress_jump
    star = host_name_list[j]
    # f-string is used to add the ./output/ and .tbl parts so that the KoA searching code can work properly.
    output = f"./output/{host_name_list[j]}.tbl"
    output_file = output.replace(" ", "_")
    Koa.query_object ('hires', \
                  star, \
                  output_file, overwrite=True,)
    
    output_dir = f"dnload_dir_hires_calib1/{host_name_list[j]}"
    output_dir_no_space = output_dir.replace(" ", "_")
    rec = Table.read (output_file, format='ascii.ipac')
    print (rec)
    Koa.download (output_file, \
        'ipac', \
        output_dir_no_space, \
        lev1file=1 )
    # Lets me know how many stars I have gotten through 
    print('Just completed star i =', j)

#print(f"'{host_name_list[0]}'")

# for i in range(len(host_name_list)):
#     dd = (host_name_list[i])
#     bb = dd.replace(" ", "_")
#     cc = f''./output/
#     print(bb)

    

object name resolved: ra= 0.1856063, dec=-54.8308228
submitting request...
Result downloaded to file [./output/HIP_65_A.tbl]
koaid ofname instrume targname object ... equinox datlevel outfile filehand
----- ------ -------- -------- ------ ... ------- -------- ------- --------
There is no data in the metadata table.
Just completed star i = 0
object name resolved: ra= 0.325761, dec=-8.9262512
submitting request...
Result downloaded to file [./output/WASP-136.tbl]
          koaid           ...
------------------------- ...
HI.20220916.31125.93.fits ...
Start downloading 1 koaid data you requested;
please check your outdir: dnload_dir_hires_calib1/WASP-136 for  progress ....
Failed to get level 1 file list for koaid: HI.20220916.31125.93.fits
level 1 file directory doesn't exist.
Failed to get level 1 data list for koaid: HI.20220916.31125.93.fits
No level 1 data found for koaid: [HI.20220916.31125.93.fits]

A total of 1 new lev0 FITS files downloaded.
0 new lev1 list downloaded.
0 new lev

KeyboardInterrupt: 

In [24]:
# Using fits file downloaded for star TOI-260

# Uses the same code from week 1

# Open the file using a context manager to ensure it closes properly
with fits.open("dnload_dir_hires_calib1/TOI-260/lev0/HI.20190820.50333.fits") as hdul:
    # See a summary of the file structure (extensions, dimensions, etc.)
    hdul.info()
    
    # Access the primary HDU (usually index 0)
    primary_hdu = hdul[0]
    
    # Access data (as a NumPy array) and header (as a dict-like object)
    data = primary_hdu.data
    header = primary_hdu.header
    
    # Access specific header keywords
    exposure_time = header['EXPTIME']

    data_table = Table(hdul[1].data)
    
    # Access columns by name
    wavelengths = data_table['WAVE'][0]
    flux = data_table['FLUX'][0]

    print(wavelengths)
    print(flux)

# The fits file does not get the wavelength data like how it was accessed in week 1, and when digging around in the fits file
# I couldn't find anything mentioning the flux.

Filename: dnload_dir_hires_calib1/TOI-260/lev0/HI.20190820.50333.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     811   ()      
  1  VidInp2       1 ImageHDU       360   (713, 4096)   int16 (rescales to uint16)   
  2  VidInp4       1 ImageHDU       360   (713, 4096)   int16 (rescales to uint16)   
  3  VidInp6       1 ImageHDU       360   (713, 4096)   int16 (rescales to uint16)   


KeyError: 'WAVE'

In [20]:
# Accesses a level 1 flux table for TOI-260 that has both flux and wavelength values

filename = 'dnload_dir_hires_calib1/TOI-260/lev1/tbl/ccd1/flux/HI.20190820.50333_1_01_flux.tbl.gz'
a = pd.read_table(filename)
print(a)

# As there is only 1 column, I am having trouble extracting the flux and wavelength data to plot.

     | col    | row    |raw_col |raw_row | wave     | Flux         | Error        | Background   | Sig_to_Noise | Flat         | Arc_Lamp     | Sum_Flux     |
0          0.00   165.83   500.17     0.00  3795.336...                                                                                                       
1          1.00   165.84   500.16     1.00  3795.352...                                                                                                       
2          2.00   165.85   500.15     2.00  3795.367...                                                                                                       
3          3.00   165.86   500.14     3.00  3795.383...                                                                                                       
4          4.00   165.88   500.12     4.00  3795.398...                                                                                                       
...                                           

    Header size is not multiple of 2880: 624778
There may be extra bytes after the last HDU or the file is corrupted. [astropy.io.fits.hdu.hdulist]


OSError: Empty or corrupt FITS file